# Dataset Analysis

This notebook covers **EDA** (one clear section per dataset) and **post-training diagnostics** (predicted vs actual, residuals, feature importance).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from src.analysis.dataset_report import print_basic_info, print_missing_values, print_target_summary
from src.analysis.plots import (
    plot_target_distribution,
    plot_numeric_vs_target,
    plot_attendance_by_category,
    plot_attendance_by_year,
    plot_correlation_heatmap,
    plot_predicted_vs_actual,
    plot_residuals,
    plot_feature_importance,
)
from src.modeling.models import get_regression_models
from src.modeling.metrics import calculate_regression_metrics

In [ ]:
DATASETS = {
    "v1_baseline":        "E:\\code\\tour-prediction\\data\\processed\\attendance_v1_baseline.csv",
    "v2_artist":          "E:\\code\\tour-prediction\\data\\processed\\attendance_v2_artist.csv",
    "v3_artist_geo":      "E:\\code\\tour-prediction\\data\\processed\\attendance_v3_artist_geo.csv",
    "v4_artist_geo_time": "E:\\code\\tour-prediction\\data\\processed\\attendance_v4_artist_geo_time.csv",
    "v5_full":            "E:\\code\\tour-prediction\\data\\processed\\attendance_v5_full.csv",
    "v6_no_artist":       "E:\\code\\tour-prediction\\data\\processed\\attendance_v6_no_artist.csv",
}

LEAKY_COLS = ["fill_rate", "box_score", "avg_ticket_price"]
TARGET = "attendance"

# Categorical columns to plot boxplots for (only plotted when present in dataset)
CAT_COLS = ["artist_name", "country", "venue_type"]

## Part 1 — Exploratory Data Analysis

One section per dataset. Each section covers:
1. Basic info + missing values + target summary
2. Target distribution (linear and log scale)
3. Numeric features vs target (scatter)
4. Categorical features vs target (box plots)
5. Attendance over time (when `show_year` is present)
6. Correlation heatmap

In [ ]:
for dataset_name, dataset_path in DATASETS.items():

    # ── Header ────────────────────────────────────────────────────────────────
    print()
    print("#" * 70)
    print(f"#  DATASET: {dataset_name}")
    print("#" * 70)

    df = pd.read_csv(dataset_path)

    # ── 1. Basic info ─────────────────────────────────────────────────────────
    print("\n── 1. Basic info " + "─" * 53)
    print_basic_info(df, dataset_name)
    print_missing_values(df)
    print_target_summary(df, target=TARGET)

    # ── 2. Target distribution ────────────────────────────────────────────────
    print("\n── 2. Target distribution " + "─" * 44)
    plot_target_distribution(df, dataset_name, target=TARGET)

    # ── 3. Numeric features vs target ─────────────────────────────────────────
    print("\n── 3. Numeric features vs target " + "─" * 37)
    plot_numeric_vs_target(df, dataset_name, target=TARGET)

    # ── 4. Categorical features vs target ─────────────────────────────────────
    print("\n── 4. Categorical features vs target " + "─" * 33)
    for cat_col in CAT_COLS:
        plot_attendance_by_category(df, dataset_name, cat_col, target=TARGET)

    # ── 5. Attendance over time ───────────────────────────────────────────────
    print("\n── 5. Attendance over time " + "─" * 43)
    plot_attendance_by_year(df, dataset_name, target=TARGET)

    # ── 6. Correlation heatmap ────────────────────────────────────────────────
    print("\n── 6. Correlation heatmap " + "─" * 44)
    plot_correlation_heatmap(df, dataset_name, target=TARGET)

    print("\n" + "─" * 70 + "\n")

## Part 2 — Post-Training Diagnostics

For each dataset × model combination:
- **Predicted vs Actual** — how well predictions track reality
- **Residuals** — where and how the model fails
- **Feature Importance** — what drives predictions (tree models only)

In [ ]:
for dataset_name, dataset_path in DATASETS.items():

    # ── Header ────────────────────────────────────────────────────────────────
    print()
    print("#" * 70)
    print(f"#  DIAGNOSTICS: {dataset_name}")
    print("#" * 70)

    # ── Prepare data (mirrors experiments.py exactly) ─────────────────────────
    df = pd.read_csv(dataset_path)

    if "reporting_status" in df.columns:
        df = df[df["reporting_status"] == "reported"]

    df = df.dropna(subset=[TARGET, "venue_capacity"])

    y = df[TARGET]
    X = df.drop(columns=[TARGET])

    cols_to_drop = [c for c in LEAKY_COLS if c in X.columns]
    X = X.drop(columns=cols_to_drop)

    X = pd.get_dummies(X, drop_first=True)
    feature_names = X.columns.tolist()

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc  = scaler.transform(X_test)

    # ── Train & diagnose each model ───────────────────────────────────────────
    models_dict = get_regression_models()

    for model_name, model in models_dict.items():
        print(f"\n── {model_name} " + "─" * (60 - len(model_name)))

        model.fit(X_train_sc, y_train)
        y_pred = model.predict(X_test_sc)

        metrics = calculate_regression_metrics(y_test, y_pred)
        print(f"  MAE={metrics['MAE']:,.0f}  RMSE={metrics['RMSE']:,.0f}  R²={metrics['R2']:.4f}")

        plot_predicted_vs_actual(y_test, y_pred, model_name, dataset_name, target=TARGET)
        plot_residuals(y_test, y_pred, model_name, dataset_name)

        # Feature importance — only for tree-based models
        if hasattr(model, "feature_importances_"):
            plot_feature_importance(
                feature_names,
                model.feature_importances_,
                model_name,
                dataset_name,
                top_n=20,
            )

    print("\n" + "─" * 70 + "\n")